# 🏠 Exploratory Data Analysis: Real-World House Price Prediction

This notebook conducts in-depth exploratory data analysis (EDA) on the residential housing dataset used for training our **Custom Regularized Linear Regression** model with **Batch Gradient Descent**.

### Key Objectives:
1. Inspect dataset structure, distributions, and missing values.
2. Examine feature correlations and identify primary price drivers.
3. Investigate geographical price variation across Indian metropolitan areas.
4. Assess outlier behavior and evaluate the rationale for IQR soft capping.
5. Confirm absence of target leakage in derived features.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Load dataset
data_path = "../data/raw/house_prices.csv"
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

## 1. Summary Statistics & Missing Values Analysis

In [ ]:
print("--- Missing Values Count ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n--- Descriptive Statistics ---")
df.describe().T[["mean", "std", "min", "50%", "max"]]

## 2. Target Variable Distribution: House Price (₹)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Untransformed price distribution (Lakhs)
sns.histplot(df["price"] / 100000.0, kde=True, ax=ax1, color="#3B82F6", bins=35)
ax1.set_title("Distribution of Property Prices (in ₹ Lakhs)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Price (₹ Lakhs)")
ax1.set_ylabel("Count")

# Price by City
sns.boxplot(data=df, x="location", y=df["price"] / 100000.0, palette="Set2", ax=ax2)
ax2.set_title("Price Distribution across Metros", fontsize=12, fontweight="bold")
ax2.set_xlabel("City")
ax2.set_ylabel("Price (₹ Lakhs)")
plt.tight_layout()
plt.show()

## 3. Floor Area vs Price by City Tier

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df.sample(min(len(df), 2000), random_state=42),
    x="area_sqft",
    y=df["price"] / 100000.0,
    hue="location",
    palette="Set1",
    alpha=0.6,
    s=40
)
plt.title("Area (sq.ft) vs. Price (₹ Lakhs) Colored by Location", fontsize=14, fontweight="bold")
plt.xlabel("Area (sq.ft)")
plt.ylabel("Price (₹ Lakhs)")
plt.show()

## 4. Correlation Matrix of Numerical Features

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True)
plt.title("Correlation Matrix of Numeric Attributes", fontsize=14, fontweight="bold", pad=12)
plt.show()

## 5. Summary Findings for Modeling

- **Area and Location Tier** are the single strongest linear drivers of property valuation.
- **Distance to City Center and Crime Rate** exhibit strong negative associations with price.
- **Feature Scaling (`StandardScaler`)** is strictly required: numerical scales range from 0.02 (crime rate) to 5,000 (area sqft) and 50,000 (property tax). Without standardization, Batch Gradient Descent would experience severe numerical instability and oscillate wildly.
- **L1/L2 Regularization** balances collinearity between bedrooms, bathrooms, and total area.